In [53]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


In [56]:
df = pd.read_csv(
    "/Users/aashishmewada/Desktop/PRISM AI/PRISM-AI/data/raw/customer .csv"
)

df.head()

,CustomerID,Age,Gender,Income,Location,Membership,RegistrationDate,Tenure,TotalSpend,TotalTransactions,...,CartAbandonmentRate,EmailOpenRate,MarketingClicks,SatisfactionScore,Rating,SupportCalls,Complaints,Reviews,CustomerHealthScore,ChurnRisk
0,CUST000001,54,Female,72543.43,NJ,Gold,2020-03-10,6.81,4389.80,30,...,48.5,65.6,79,65,3.2,0,2,8.0,31.8,High
1,CUST000002,42,Male,78149.18,AZ,Silver,2021-11-10,5.14,706.89,2,...,87.9,23.4,0,1,0.0,5,18,0.0,11.6,High
2,CUST000003,24,Female,48635.85,MD,Free,2026-08-18,0.37,2935.65,20,...,87.8,62.3,24,55,2.8,7,1,23.0,18.2,High
3,CUST000004,31,Male,19935.67,GA,Free,2025-11-20,1.11,1410.68,32,...,73.0,32.0,23,74,3.7,5,3,3.0,21.0,High
4,CUST000005,45,Female,83500.57,AZ,Silver,2020-03-11,6.80,4352.92,69,...,25.5,59.0,38,77,3.8,4,2,17.0,36.4,High


In [57]:
df.shape

(50000, 28)

In [58]:
df.isnull().sum()

CustomerID                0
Age                       0
Gender                    0
Income                 2236
Location                  0
Membership                0
RegistrationDate          0
Tenure                    0
TotalSpend                0
TotalTransactions         0
AverageOrderValue         0
PurchaseFrequency         0
LastPurchaseDays          0
PreferredCategory         0
WebsiteVisits             0
AppUsageMinutes           0
LoginFrequency            0
WishlistCount          1110
CartAbandonmentRate    1253
EmailOpenRate          2402
MarketingClicks           0
SatisfactionScore         0
Rating                 1824
SupportCalls              0
Complaints                0
Reviews                2375
CustomerHealthScore       0
ChurnRisk                 0
dtype: int64

In [59]:
num_cols = df.select_dtypes(
    include=np.number
).columns


for col in num_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )

In [60]:
cat_cols = df.select_dtypes(
    include="object"
).columns


for col in cat_cols:
    df[col] = df[col].fillna(
        df[col].mode()[0]
    )

/var/folders/rf/55f17gkd0fs1mt6d7fkb_x340000gn/T/ipykernel_3423/1214591063.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(


In [61]:
df.isnull().sum().sum()

np.int64(0)

In [62]:
df_model = df.drop(
    [
        "CustomerID",
        "RegistrationDate"
    ],
    axis=1
)

In [63]:
encoder = LabelEncoder()


for col in df_model.select_dtypes(
    include="object"
).columns:
    
    df_model[col] = encoder.fit_transform(
        df_model[col]
    )

/var/folders/rf/55f17gkd0fs1mt6d7fkb_x340000gn/T/ipykernel_3423/1610245923.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_model.select_dtypes(


In [65]:
X = df_model.drop(
    "ChurnRisk",
    axis=1
)


y = df_model["ChurnRisk"]


In [66]:
y.value_counts()

ChurnRisk
0    37978
2     6048
1     5974
Name: count, dtype: int64

In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [68]:
scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(
    X_train
)


X_test_scaled = scaler.transform(
    X_test
)

In [72]:
models = {

"Logistic Regression":
LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
),


"Decision Tree":
DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
),


"Random Forest":
RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
),


"Gradient Boosting":
GradientBoostingClassifier(
    random_state=42
)

}

In [73]:
from sklearn.metrics import *

results = []


for name, model in models.items():

    model.fit(
        X_train_scaled,
        y_train
    )

    pred = model.predict(
        X_test_scaled
    )

    prob = model.predict_proba(
        X_test_scaled
    )[:,1]


    results.append({

        "Model": name,

        "Accuracy":
        accuracy_score(
            y_test,
            pred
        ),

        "Precision":
        precision_score(
            y_test,
            pred,
            zero_division=0
        ),

        "Recall":
        recall_score(
            y_test,
            pred,
            zero_division=0
        ),

        "F1":
        f1_score(
            y_test,
            pred,
            zero_division=0
        ),

        "ROC-AUC":
        roc_auc_score(
            y_test,
            prob
        )
    })


result_df = pd.DataFrame(results)

result_df.sort_values(
    "ROC-AUC",
    ascending=False
)

ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

In [74]:
print(y.unique())

[0 2 1]


In [75]:
print(y.value_counts())

ChurnRisk
0    37978
2     6048
1     5974
Name: count, dtype: int64


In [76]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


results = []


for name, model in models.items():

    model.fit(
        X_train_scaled,
        y_train
    )

    pred = model.predict(
        X_test_scaled
    )


    results.append({

        "Model": name,

        "Accuracy":
        accuracy_score(
            y_test,
            pred
        ),

        "Precision":
        precision_score(
            y_test,
            pred,
            average="weighted"
        ),

        "Recall":
        recall_score(
            y_test,
            pred,
            average="weighted"
        ),

        "F1 Score":
        f1_score(
            y_test,
            pred,
            average="weighted"
        )
    })


result_df = pd.DataFrame(results)

result_df.sort_values(
    "F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
1,Decision Tree,1.0000,1.000000,1.0000,1.000000
2,Random Forest,1.0000,1.000000,1.0000,1.000000
3,Gradient Boosting,1.0000,1.000000,1.0000,1.000000
0,Logistic Regression,0.9842,0.985806,0.9842,0.984574


In [77]:
from sklearn.metrics import roc_auc_score

In [78]:
roc_auc_score(
    y_test,
    model.predict_proba(X_test_scaled),
    multi_class="ovr"
)

1.0

In [79]:
print(y_train.unique())

[0 2 1]


In [81]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)
from sklearn.tree import DecisionTreeClassifier

In [82]:
models = {

    "Logistic Regression":
    LogisticRegression(
        max_iter=1000
    ),

    "Random Forest":
    RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting":
    GradientBoostingClassifier(
        random_state=42
    ),

    "Extra Trees":
    ExtraTreesClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Decision Tree":
    DecisionTreeClassifier(
        random_state=42
    )
}

In [83]:
from sklearn.metrics import *

results=[]


for name,model in models.items():

    print("Training:",name)

    model.fit(
        X_train_scaled,
        y_train
    )

    pred = model.predict(
        X_test_scaled
    )


    results.append({

        "Model":name,

        "Accuracy":
        accuracy_score(
            y_test,
            pred
        ),

        "Precision":
        precision_score(
            y_test,
            pred,
            average="weighted"
        ),

        "Recall":
        recall_score(
            y_test,
            pred,
            average="weighted"
        ),

        "F1 Score":
        f1_score(
            y_test,
            pred,
            average="weighted"
        )

    })


result_df = pd.DataFrame(results)


result_df.sort_values(
    "F1 Score",
    ascending=False
)

Training: Logistic Regression
Training: Random Forest
Training: Gradient Boosting
Training: Extra Trees
Training: Decision Tree


,Model,Accuracy,Precision,Recall,F1 Score
2,Gradient Boosting,1.0000,1.000000,1.0000,1.000000
4,Decision Tree,1.0000,1.000000,1.0000,1.000000
1,Random Forest,0.9999,0.999900,0.9999,0.999900
0,Logistic Regression,0.9939,0.993881,0.9939,0.993886
3,Extra Trees,0.9908,0.990858,0.9908,0.990670


In [86]:
best_model = RandomForestClassifier(...)

In [87]:
importance = pd.DataFrame({

"Feature":X.columns,

"Importance":
best_model.feature_importances_

})


importance.sort_values(
"Importance",
ascending=False
)

NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.